## pickle → SCIDAC converter

Converts qlat topological charge density fields (stored as numpy arrays in a pickle)
into SCIDAC/LIME files readable by Grid's `ScidacReader`.

**Axis convention**:  
qlat stores fields in `(x, y, z, t)` order (x slowest).  
Grid/SCIDAC expects `(t, z, y, x)` order (t slowest, x fastest) in the binary payload.  
The conversion is `.T` (reverses all 4 axes).

**Source**: `32c-hmc-test-gen.ipynb` reads `f_tadpole_loop.field` from qlat  
(`rand_vol_u1_idx-0` and `-1`: two stochastic U(1) estimates of the DWF midpoint TCD, i.e. `q_A`).  
Config: trajectory 702, 32⁴ lattice.

In [ ]:
import struct
import pickle
import numpy as np
import os

In [ ]:
# ── LIME record writer ────────────────────────────────────────────────────────
LIME_MAGIC   = 0x456789AB01234567
LIME_VERSION = 1

def _lime_record(type_str: str, data: bytes, MB: bool, ME: bool) -> bytes:
    """Pack one LIME record (148-byte header + data padded to 8-byte boundary)."""
    flags  = (0x8000 if MB else 0) | (0x4000 if ME else 0)
    type_b = type_str.encode('ascii').ljust(128, b'\x00')[:128]
    header = struct.pack('>QHHQ128s', LIME_MAGIC, LIME_VERSION, flags, len(data), type_b)
    pad    = (8 - len(data) % 8) % 8
    return header + data + b'\x00' * pad

In [ ]:
# ── SCIDAC writer for a complex scalar field ──────────────────────────────────
def write_scidac_complex(fname: str, field: np.ndarray):
    """
    Write a 4D complex scalar field to a SCIDAC/LIME file.

    Parameters
    ----------
    fname : output filename
    field : numpy array, shape (Nt, Nz, Ny, Nx), dtype complex128
            Grid order: t slowest, x fastest.
    """
    assert field.ndim == 4, "field must be 4D"
    Nt, Nz, Ny, Nx = field.shape

    # Record 1: scidac-file-xml  (standalone message)
    file_xml = (
        '<?xml version="1.0"?>'
        '<scidacFile>'
          '<version>1.1</version>'
          '<spacetime>4</spacetime>'
          f'<dims>{Nx} {Ny} {Nz} {Nt}</dims>'
          '<volfmt>0</volfmt>'
        '</scidacFile>'
    ).encode()
    rec1 = _lime_record('scidac-file-xml', file_xml, MB=True, ME=True)

    # Record 2: scidac-record-xml  (begins field message)
    rec_xml = (
        '<?xml version="1.0"?>'
        '<scidacRecord>'
          '<version>1.0</version>'
          '<date></date>'
          '<globaldata>0</globaldata>'
          '<datatype>4D_COMPLEX_DOUBLE</datatype>'
          '<precision>D</precision>'
          '<colors>1</colors>'
          '<spins>1</spins>'
          '<typesize>16</typesize>'
          '<datacount>1</datacount>'
        '</scidacRecord>'
    ).encode()
    rec2 = _lime_record('scidac-record-xml', rec_xml, MB=True, ME=False)

    # Record 3: scidac-binary-data  (ends field message)
    # Interleave re/im and byte-swap to big-endian
    flat = np.ascontiguousarray(field, dtype=np.complex128).ravel()
    be   = np.empty(2 * len(flat), dtype='>f8')
    be[0::2] = flat.real
    be[1::2] = flat.imag
    rec3 = _lime_record('scidac-binary-data', be.tobytes(), MB=False, ME=True)

    with open(fname, 'wb') as f:
        f.write(rec1)
        f.write(rec2)
        f.write(rec3)
    print(f'  wrote {fname}  ({os.path.getsize(fname)/1024:.0f} kB)')

In [ ]:
# ── Load pickle ───────────────────────────────────────────────────────────────
pickle_path = '../../data/32c-hmc-test-demo.pickle'   # adjust if needed
out_dir     = 'data'
os.makedirs(out_dir, exist_ok=True)

with open(pickle_path, 'rb') as f:
    data = pickle.load(f)

print('Keys:', list(data.keys()))
for k, v in data.items():
    print(f'  {k}: shape={v.shape}  dtype={v.dtype}')

In [ ]:
# ── Convert and write ─────────────────────────────────────────────────────────
for key, arr in data.items():
    print(f'\n{key}:')
    print(f'  Q (real) = {arr.real.sum():.6f}')
    print(f'  imag rms = {np.sqrt(np.mean(arr.imag**2)):.3e}')

    # qlat: (x,y,z,t) x-slowest  →  Grid: (t,z,y,x) t-slowest, x-fastest
    field_grid = arr.T    # .T reverses all 4 axes

    out_path = os.path.join(out_dir, f'{key}.scidac')
    write_scidac_complex(out_path, field_grid)

In [ ]:
# ── Sanity check: RMS difference between the two stochastic estimates ─────────
# topo_field_0 and topo_field_1 are two independent U(1) source estimates of q_A.
# Their RMS difference / sqrt(2) is the per-estimator stochastic noise.
f0 = data['topo_field_0']
f1 = data['topo_field_1']
rms_diff = np.sqrt(np.sum((f1 - f0).real**2) / 2)
print(f'RMS(field_1 - field_0) / sqrt(2) = {rms_diff:.6f}  (stochastic noise per estimate)')
print(f'Noise / |Q|             = {rms_diff / abs(f0.real.sum()):.4f}')